In [2]:
# =========================
# FULL PIPELINE (1 CELL)
# =========================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("train.csv")  # sesuaikan path

# =========================
# 2. PREPROCESSING
# =========================

# Handle missing
df['previous_year_rating'].fillna(df['previous_year_rating'].median(), inplace=True)

# =========================
# HANDLE MISSING VALUES (FULL FIX)
# =========================

# numerical
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

# categorical
cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Encode categorical
cat_cols = ['department', 'region', 'education', 'gender', 'recruitment_channel']
le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col])

# =========================
# 3. SPLIT DATA
# =========================
X = df.drop(columns=['employee_id', 'is_promoted'])
y = df['is_promoted']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# 4. HANDLE IMBALANCE (SMOTE)
# =========================
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

# =========================
# 5. SCALING
# =========================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =========================
# 6. TRAIN MODEL (XGBOOST)
# =========================
model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

# =========================
# 7. EVALUASI
# =========================
y_pred = model.predict(X_test)
print("=== Classification Report ===")
print(classification_report(y_test, y_pred))

# =========================
# 8. AMBIL PROBABILITAS
# =========================
y_proba = model.predict_proba(X_test)[:, 1]

df_test = pd.DataFrame(X_test, columns=X.columns)
df_test['ml_score'] = y_proba
df_test['actual'] = y_test.values

# =========================
# 9. SIAPKAN DATA SPK (SAW)
# =========================
decision_df = df_test[['ml_score', 'avg_training_score', 'length_of_service', 'awards_won?']].copy()

# Normalisasi (benefit semua)
decision_norm = decision_df / decision_df.max()

# =========================
# 10. BOBOT (AHP sederhana)
# =========================
weights = {
    'ml_score': 0.4,
    'avg_training_score': 0.3,
    'length_of_service': 0.2,
    'awards_won?': 0.1
}

# =========================
# 11. HITUNG SKOR SAW
# =========================
decision_norm['score'] = (
    decision_norm['ml_score'] * weights['ml_score'] +
    decision_norm['avg_training_score'] * weights['avg_training_score'] +
    decision_norm['length_of_service'] * weights['length_of_service'] +
    decision_norm['awards_won?'] * weights['awards_won?']
)

# Ranking
decision_norm['rank'] = decision_norm['score'].rank(ascending=False)

# =========================
# 12. OUTPUT
# =========================
print("\n=== TOP 10 KANDIDAT PROMOSI ===")
print(decision_norm.sort_values(by='score', ascending=False).head(10))

C:\Users\ASUS\AppData\Local\Temp\ipykernel_18648\926167923.py:25: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['previous_year_rating'].fillna(df['previous_year_rating'].median(), inplace=True)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_18648\926167923.py:35: ChainedAssignmentError: A value is being set on a copy of a DataFrame or

ValueError: Input X contains NaN.
SMOTE does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values